# Method 4 - k-means + Stacked Autoencoder

Three sequentially-trained autoencoders (hierarchical features) followed by
k-means on the final 2-D latent. See `src.autoencoders.build_stacked_features`
for the exact hierarchy (matches the paper).

**What changed vs the original**
- Shared `src/` preprocessing (this method previously used a different `>= 0`
  fill mask, so it clustered a *different* pixel set than the others).
- Stacked training centralised and **seeded/deterministic**.
- Shared majority filter + `finalize_method`; intrinsic metrics on the 2-D latent.

In [ ]:
# Make the shared src/ package importable regardless of CWD.
import sys, os
sys.path.insert(0, os.path.abspath("."))

import numpy as np
import matplotlib.pyplot as plt

from src import config
from src.seeding import seed_everything
from src.preprocessing import load_and_preprocess_enmap
from src.results import finalize_method
from src.postprocess import majority_filter

seed_everything()          # seed numpy / random / torch from config.SEED
print(f"Seed = {config.SEED}")

In [ ]:
from sklearn.cluster import KMeans
from src.autoencoders import build_stacked_features, get_device

res = load_and_preprocess_enmap(config.ENMAP_TIF, standardize=False)
X = res.data_valid
device = get_device(); print("device:", device)

latent, histories = build_stacked_features(X, seed=config.SEED, device=device)
print("final latent:", latent.shape)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (k, h) in zip(axes, histories.items()):
    ax.plot(h["loss"]); ax.set_title(f"{k} loss"); ax.set_xlabel("epoch")
plt.tight_layout(); plt.savefig(config.FIGURES_DIR / "sa_training.png", dpi=200); plt.show()

In [ ]:
km = KMeans(n_clusters=config.KMEANS_K, random_state=config.SEED, n_init=config.KMEANS_N_INIT)
labels_raw = km.fit_predict(latent)

sa_map = finalize_method("sa", res, labels_raw, features_for_intrinsic=latent)

def plot_cluster_map(labels_spatial, extent, title):
    n = int(np.nanmax(labels_spatial)) + 1 if np.isfinite(np.nanmax(labels_spatial)) else 1
    fig, ax = plt.subplots(figsize=(12, 9))
    cmap = plt.get_cmap("rainbow", n).copy()
    cmap.set_bad(color="white")
    im = ax.imshow(labels_spatial, cmap=cmap, extent=extent, interpolation="nearest")
    cbar = plt.colorbar(im, ax=ax, label="Cluster", fraction=0.046)
    ax.set_xlabel("Easting (m)"); ax.set_ylabel("Northing (m)")
    ax.set_title(title, fontsize=13)
    plt.tight_layout()
    return fig, ax

plot_cluster_map(sa_map, res.extent, f"k-means + Stacked AE (k={config.KMEANS_K})")
plt.savefig(config.FIGURES_DIR / "sa_classification.png", dpi=200, bbox_inches="tight"); plt.show()